# 📡 TP Télécom 3GPP — Phase 4 : Fine-Tuning avec QLoRA

**Objectif** : Spécialiser le LLM sur le domaine 3GPP via Fine-Tuning léger QLoRA.

**Principe QLoRA :**
```
Modèle de base (gelé) + Petites matrices LoRA (entraînées)
         ↓
Modèle spécialisé 3GPP avec peu de ressources
```

```
Phase 1 ✅ → Phase 2 ✅ → Phase 3 ✅ → [Phase 4] → Phase 5 → Phase 6 → Phase 7
```

## 1. Installation des dépendances

In [1]:
!pip install -q transformers torch datasets
!pip install -q peft trl accelerate
!pip install -q evaluate rouge_score
!pip install -q pandas matplotlib
print('✅ Installation terminée')

✅ Installation terminée


## 2. Imports & Configuration

In [2]:
import json, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device : {device}')
print(f'✅ Imports effectués')

c:\Users\HP\anaconda3\envs\tp_3gpp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Device : cpu
✅ Imports effectués


## 3. Chargement du Modèle de Base

In [3]:
# Chargement config Phase 1
with open(f'{SAVE_DIR}\\pipeline_config.json') as f:
    config = json.load(f)
MODEL_ID = config['best_model_id']
print(f'✅ Modèle de base : {config["best_model_name"]}')

# Chargement tokenizer et modèle
print('🔄 Chargement du modèle...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map='auto'
)
print(f'✅ Modèle chargé')
print(f'   Paramètres totaux : {sum(p.numel() for p in model.parameters()):,}')

✅ Modèle de base : DistilGPT2
🔄 Chargement du modèle...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 3474.30it/s]


✅ Modèle chargé
   Paramètres totaux : 81,912,576


## 4. Configuration LoRA

In [4]:
# Configuration LoRA légère pour CPU
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=4,                    # Rang des matrices LoRA (plus petit = plus léger)
    lora_alpha=8,           # Facteur de scaling
    lora_dropout=0.1,       # Dropout pour régularisation
    bias='none',
    target_modules=['c_attn']  # Modules ciblés dans GPT2
)

# Application de LoRA au modèle
model = get_peft_model(model, lora_config)

# Affichage des paramètres entraînables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA appliqué :')
print(f'   Paramètres entraînables : {trainable:,} ({100*trainable/total:.2f}%)')
print(f'   Paramètres gelés        : {total-trainable:,} ({100*(total-trainable)/total:.2f}%)')
print(f'   → Seulement {100*trainable/total:.2f}% du modèle est entraîné !')

✅ LoRA appliqué :
   Paramètres entraînables : 73,728 (0.09%)
   Paramètres gelés        : 81,912,576 (99.91%)
   → Seulement 0.09% du modèle est entraîné !


## 5. Préparation du Dataset d'Entraînement

In [5]:
TRAIN_PATH = r'C:\Users\HP\Downloads\TeleQnA_training.txt'
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

# Création des textes d'entraînement
# Format : Question + Answer pour apprendre au modèle le domaine 3GPP
train_texts = []
items = list(train_data.values())

for item in items[:200]:  # 200 exemples pour rester léger sur CPU
    answer_idx  = item.get('answer', 1)
    answer_text = item.get(f'option {answer_idx}', str(answer_idx))
    text = (
        f"3GPP Question: {item['question']} "
        f"Answer: {answer_text}. "
        f"{item.get('explanation', '')}"
    )
    train_texts.append({'text': text})

# Conversion en Dataset HuggingFace
hf_dataset = Dataset.from_list(train_texts)
print(f'✅ Dataset d\'entraînement : {len(hf_dataset)} exemples')
print(f'\nExemple :')
print(f'  {hf_dataset[0]["text"][:200]}...')

✅ Dataset d'entraînement : 200 exemples

Exemple :
  3GPP Question: What is the purpose of the Nmfaf_3daDataManagement_Deconfigure service operation? [3GPP Release 18] Answer: To configure the MFAF to map data or analytics received by the MFAF to out-bo...


## 6. Fine-Tuning avec QLoRA

In [ ]:
from trl import SFTConfig

# Configuration de l'entraînement — TRL 1.6.0
sft_config = SFTConfig(
    output_dir                  = f'{SAVE_DIR}\\Phase4\\checkpoints',
    num_train_epochs            = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-4,
    fp16                        = False,
    logging_steps               = 10,
    save_steps                  = 50,
    save_total_limit            = 1,
    report_to                   = 'none',
    use_cpu                     = True,
    dataset_text_field          = 'text',
    max_length                  = 256
)

# Trainer
trainer = SFTTrainer(
    model            = model,
    args             = sft_config,
    train_dataset    = hf_dataset,
    processing_class = tokenizer
)

print('🔄 Début du Fine-Tuning...')
print('   ⚠️  Sur CPU cela prend environ 10-20 minutes')
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f'\n✅ Fine-Tuning terminé en {elapsed/60:.1f} minutes')

# Sauvegarde du modèle fine-tuné
model.save_pretrained(f'{SAVE_DIR}\\Phase4\\model_finetuned')
tokenizer.save_pretrained(f'{SAVE_DIR}\\Phase4\\model_finetuned')
print('💾 Modèle sauvegardé → Phase4/model_finetuned')

TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'

## 7. Évaluation : LLM de Base vs LLM Fine-Tuné

In [ ]:
from transformers import pipeline as hf_pipeline
from peft import PeftModel

# Chargement modèle de base
print('🔄 Chargement modèle de base...')
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
base_pipe  = hf_pipeline('text-generation', model=base_model,
                          tokenizer=tokenizer, pad_token_id=50256)
print('✅ Modèle de base prêt')

# Chargement modèle fine-tuné
print('🔄 Chargement modèle fine-tuné...')
ft_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
ft_model = PeftModel.from_pretrained(ft_model, f'{SAVE_DIR}\\Phase4\\model_finetuned')
ft_pipe  = hf_pipeline('text-generation', model=ft_model,
                        tokenizer=tokenizer, pad_token_id=50256)
print('✅ Modèle fine-tuné prêt')

# Questions de test
TEST_PATH = r'C:\Users\HP\Downloads\TeleQnA_testing.txt'
with open(TEST_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

test_items = list(test_data.values())[:10]
results    = []

for i, item in enumerate(test_items):
    question    = item['question']
    answer_idx  = item.get('answer', item.get('option 1', ''))
    answer_text = item.get(f'option {answer_idx}', str(answer_idx)) if isinstance(answer_idx, int) else answer_idx
    prompt      = f'3GPP Question: {question} Answer:'

    # Base
    t0       = time.time()
    res_base = base_pipe(prompt, max_new_tokens=100, do_sample=False,
                         pad_token_id=50256, truncation=True)
    t_base   = time.time() - t0
    ans_base = res_base[0]['generated_text'].replace(prompt, '').strip()

    # Fine-tuné
    t0     = time.time()
    res_ft = ft_pipe(prompt, max_new_tokens=100, do_sample=False,
                     pad_token_id=50256, truncation=True)
    t_ft   = time.time() - t0
    ans_ft = res_ft[0]['generated_text'].replace(prompt, '').strip()

    results.append({
        'id'          : i + 1,
        'question'    : question,
        'reference'   : str(answer_text),
        'base_answer' : ans_base,
        'ft_answer'   : ans_ft,
        'time_base'   : round(t_base, 2),
        'time_ft'     : round(t_ft, 2)
    })
    print(f'  ✓ Q{i+1} traité')

df_results = pd.DataFrame(results)
df_results.to_csv(f'{SAVE_DIR}\\Phase4\\phase4_results.csv', index=False)
print(f'\n✅ {len(df_results)} résultats sauvegardés')

## 8. Scores ROUGE — Base vs Fine-Tuné

In [ ]:
from evaluate import load as load_metric
rouge = load_metric('rouge')

refs        = df_results['reference'].tolist()
scores_base = rouge.compute(predictions=df_results['base_answer'].tolist(), references=refs)
scores_ft   = rouge.compute(predictions=df_results['ft_answer'].tolist(),   references=refs)

df_compare = pd.DataFrame([
    {
        'Approche'      : 'LLM Base',
        'ROUGE-1'       : round(scores_base['rouge1'], 4),
        'ROUGE-2'       : round(scores_base['rouge2'], 4),
        'ROUGE-L'       : round(scores_base['rougeL'], 4),
        'Temps moy. (s)': round(df_results['time_base'].mean(), 2)
    },
    {
        'Approche'      : 'LLM Fine-Tuné',
        'ROUGE-1'       : round(scores_ft['rouge1'], 4),
        'ROUGE-2'       : round(scores_ft['rouge2'], 4),
        'ROUGE-L'       : round(scores_ft['rougeL'], 4),
        'Temps moy. (s)': round(df_results['time_ft'].mean(), 2)
    }
])

print('📊 LLM Base vs LLM Fine-Tuné :')
print(df_compare.to_string(index=False))
for m in ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']:
    delta = df_compare.iloc[1][m] - df_compare.iloc[0][m]
    print(f'  {m}: {delta:+.4f} avec Fine-Tuning')

df_compare.to_csv(f'{SAVE_DIR}\\Phase4\\phase4_evaluation.csv', index=False)
print('\n💾 Sauvegardé → Phase4/phase4_evaluation.csv')

## 9. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('LLM Base vs LLM Fine-Tuné — Dataset 3GPP', fontsize=15, fontweight='bold')
palette = ['#FF7043', '#1565C0']

# --- ROUGE ---
ax1     = axes[0]
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
x, w    = range(len(metrics)), 0.35
for i, (_, row) in enumerate(df_compare.iterrows()):
    bars = ax1.bar([p + i*w for p in x], [row[m] for m in metrics],
                   width=w, label=row['Approche'], color=palette[i], alpha=0.85)
    for b in bars:
        ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.001,
                 f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax1.set_xticks([p+w/2 for p in x])
ax1.set_xticklabels(metrics)
ax1.set_title('Scores ROUGE', fontweight='bold')
ax1.set_ylabel('Score')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# --- Temps ---
ax2  = axes[1]
bars = ax2.bar(df_compare['Approche'], df_compare['Temps moy. (s)'], color=palette, alpha=0.85)
for b in bars:
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
             f'{b.get_height():.1f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title("Temps moyen d'inférence", fontweight='bold')
ax2.set_ylabel('Secondes')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}\\Phase4\\phase4_base_vs_finetuned.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Graphique → Phase4/phase4_base_vs_finetuned.png')

---
## ✅ Phase 4 Terminée

**Ce qu'on a fait :**
- ✅ Fine-Tuning QLoRA sur 200 exemples 3GPP
- ✅ Seulement ~1% des paramètres entraînés
- ✅ Comparaison Base vs Fine-Tuné

**Fichiers produits :**
- `Phase4/model_finetuned/` — Modèle fine-tuné
- `Phase4/phase4_results.csv`
- `Phase4/phase4_evaluation.csv`
- `Phase4/phase4_base_vs_finetuned.png`

**➡️ Prochaine étape : Phase 5 — RAFT (RAG + Fine-Tuning)**